<a href="https://colab.research.google.com/github/Symon-CNA/CNA/blob/main/PPC_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
import matplotlib.pyplot as plt

# 1. DEFINIÇÃO DAS FUNÇÕES GOVERNANTES E SOLUÇÕES ANALÍTICAS
def f_rhs(t, v, St, Res):
    """Termo da direita da EDO adimensional"""
    return (1.0 / St) * (1.0 - v - (3.0 / 8.0) * Res * (v**2))

def solucao_analitica_stokes(t, St):
    """Re -> 0 (Regime de Stokes)"""
    return 1.0 - math.exp(-t / St)

def solucao_analitica_oseen(t, St, Res):
    """Re > 0 (Ricatti / Correção de Oseen)"""
    if Res == 0:
        return solucao_analitica_stokes(t, St)

    v_star = (-1.0 + math.sqrt(1.0 + 1.5 * Res)) / (0.75 * Res)
    Q = (3.0 * Res) / (8.0 * St)
    P = - (0.75 / St) * Res * v_star - (1.0 / St)

    termo_exp = math.exp(-P * t)
    denominador = (Q / P) + ((-1.0 / v_star) - (Q / P)) * termo_exp

    return v_star + (1.0 / denominador)

# 2. IMPLEMENTAÇÃO DO MÉTODO DE RUNGE-KUTTA DE 4ª ORDEM (RK4)
def rk4_step(t, v, h, St, Res):
    """Avança um passo temporal h"""
    k1 = f_rhs(t, v, St, Res)
    k2 = f_rhs(t + 0.5 * h, v + 0.5 * h * k1, St, Res)
    k3 = f_rhs(t + 0.5 * h, v + 0.5 * h * k2, St, Res)
    k4 = f_rhs(t + h, v + h * k3, St, Res)

    v_next = v + (h / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)
    return v_next

def resolver_simulacao(t_max, h, St, Res, v0=0.0):
    """Loop principal de integração no tempo."""
    n_steps = int(math.ceil(t_max / h))
    t_list = [0.0] * (n_steps + 1)
    v_list = [0.0] * (n_steps + 1)

    for i in range(n_steps):
        t_curr = t_list[i]
        v_curr = v_list[i]

        v_next = rk4_step(t_curr, v_curr, h, St, Res)

        t_list[i+1] = t_curr + h
        v_list[i+1] = v_next

    return t_list, v_list

# 3. ANÁLISES COM EXIBIÇÃO DE VALORES E GRÁFICOS
def analisar_stokes():
    print("\n" + "="*70)
    print("1. ANÁLISE DO REGIME DE STOKES (Re -> 0) PARA DIFERENTES VALORES DE St")
    print("="*70)

    St_values = [0.1, 0.5, 1.0, 2.0]
    h = 0.01
    t_max = 10.0

    plt.figure(figsize=(9, 5))

    for St in St_values:
        t_num, v_num = resolver_simulacao(t_max, h, St, Res=0.0)
        v_ana = [solucao_analitica_stokes(t, St) for t in t_num]

        # Plota as curvas
        plt.plot(t_num, v_num, label=f'RK4 (St={St})', linestyle='-', linewidth=2)
        plt.plot(t_num, v_ana, label=f'Exata (St={St})', linestyle=':', color='black', alpha=0.6)

        # Exibe os valores numéricos no terminal
        print(f"\n--- Resultados para St = {St} (h = {h}) ---")
        print(f"{'Tempo (t*)':<12} | {'RK4 (v*)':<12} | {'Exata (v*)':<12} | {'Erro Absoluto':<15}")
        print("-" * 58)

        tempos_alvo = [0.0, 0.5, 1.0, 2.0, 5.0, 10.0]
        for t_alvo in tempos_alvo:
            idx = int(round(t_alvo / h))
            if idx < len(t_num):
                v_n = v_num[idx]
                v_a = v_ana[idx]
                erro = abs(v_n - v_a)
                print(f"{t_num[idx]:<12.2f} | {v_n:<12.6f} | {v_a:<12.6f} | {erro:<15.2e}")

    plt.title("Regime de Stokes (Re -> 0): Comparação Numérica vs Analítica", fontsize=12)
    plt.xlabel("Tempo Adimensional (t*)", fontsize=10)
    plt.ylabel("Velocidade Adimensional (v*)", fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig("grafico_stokes.png", dpi=300)
    plt.show()  # <--- Abre a janela do gráfico

def analisar_refinamento():
    print("\n" + "="*70)
    print("2. ANÁLISE DO REFINAMENTO TEMPORAL (VARIANDO PASSO h)")
    print("="*70)

    St = 1.0
    Res = 0.5
    t_max = 5.0
    h_values = [0.5, 0.2, 0.05, 0.01]

    plt.figure(figsize=(9, 5))

    # Solução analítica contínua para comparação
    t_exact = [i * 0.005 for i in range(int(t_max / 0.005) + 1)]
    v_exact = [solucao_analitica_oseen(t, St, Res) for t in t_exact]
    plt.plot(t_exact, v_exact, 'k--', label='Exata (Oseen)', linewidth=2)

    for h in h_values:
        t_num, v_num = resolver_simulacao(t_max, h, St, Res)
        plt.plot(t_num, v_num, marker='o', markersize=4, label=f'RK4 (h = {h})')

        # Calcular o erro no tempo t* = 1.0
        idx_1 = int(round(1.0 / h))
        v_num_1 = v_num[idx_1]
        v_exata_1 = solucao_analitica_oseen(1.0, St, Res)
        erro_1 = abs(v_num_1 - v_exata_1)
        print(f"Passo h = {h:<4} | v*(t*=1.0) Numérico = {v_num_1:.6f} | v* Exato = {v_exata_1:.6f} | Erro = {erro_1:.2e}")

    plt.title("Efeito do Refinamento do Passo Temporal (h)", fontsize=12)
    plt.xlabel("Tempo Adimensional (t*)", fontsize=10)
    plt.ylabel("Velocidade Adimensional (v*)", fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.savefig("grafico_refinamento.png", dpi=300)
    plt.show()  # <--- Abre a janela do gráfico

def analisar_reynolds():
    print("\n" + "="*70)
    print("3. ANÁLISE DO EFEITO INERCIAL (VARIANDO Re_s)")
    print("="*70)

    St = 1.0
    Res_values = [0.0, 0.2, 0.5, 0.8, 1.0]
    h = 0.01
    t_max = 6.0

    plt.figure(figsize=(9, 5))

    print(f"{'Re_s':<8} | {'Velocidade Terminal Limite (t* -> inf)':<35}")
    print("-" * 48)

    for Res in Res_values:
        t_num, v_num = resolver_simulacao(t_max, h, St, Res)
        v_ana = [solucao_analitica_oseen(t, St, Res) for t in t_num]

        plt.plot(t_num, v_num, label=f'Re_s = {Res}')
        plt.plot(t_num, v_ana, 'k:', alpha=0.4)

        v_terminal = v_num[-1]
        print(f"{Res:<8.1f} | {v_terminal:<35.6f}")

    plt.title("Influência das Forças Inerciais de Oseen (Re_s)", fontsize=12)
    plt.xlabel("Tempo Adimensional (t*)", fontsize=10)
    plt.ylabel("Velocidade Adimensional (v*)", fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.savefig("grafico_reynolds.png", dpi=300)
    plt.show()  # <--- Abre a janela do gráfico

if __name__ == "__main__":
    analisar_stokes()
    analisar_refinamento()
    analisar_reynolds()